# Notebook 22 — Privacy, Utility, and Adoption
## Complexity as a Third Dimension in Geoprivacy Mechanism Selection

NB20 compared seven geographic privacy mechanisms on two axes: utility (EDD) and
privacy (spatial and compound re-identification attack success). Mechanism choice in
practice, however, is also shaped by a third axis: **implementation complexity** —
the barrier between knowing a mechanism exists and actually deploying it. High
complexity reduces adoption; low adoption means no protection at all.

This notebook asks the question NB20 left open: when accounting for complexity, which
mechanisms are realistic for a public-health practitioner, and is a simpler mechanism
that gets deployed actually better than a sophisticated one that does not?

> **Synthetic data disclaimer:** All attack results in this notebook draw on the
> 489-individual Soho cholera dataset from NB14 (synthetic individual records
> parameterised from published public health statistics). They are not real patient
> records and should not be interpreted as such.

**Five-part structure:**

- **Part 1** — The Better-Than-Nothing Baseline: no-protection vs. the protection gradient
- **Part 2** — Complexity as a Third Axis: scoring seven mechanisms on implementation burden
- **Part 3** — The Three-Axis View: where each mechanism sits in privacy-utility-complexity space
- **Part 4** — Open Questions for the Field
- **Part 5** — Future Directions

<div style="background:#f5faf9;border:1px solid #b8ddd8;border-radius:8px;padding:12px 14px 14px;margin:10px 0 22px;font-family:sans-serif;">
<div style="font-size:11px;color:#5a9e99;margin-bottom:10px;font-style:italic;">Extends the NB20 baseline comparison with adoption context</div>
<div style="display:flex;align-items:stretch;flex-wrap:wrap;gap:4px;">
    <div style="background:#2a9d8f;color:white;padding:8px 14px;border-radius:4px;font-size:11px;font-weight:700;">NB20 Baseline Comparison</div>
    <div style="background:#2a9d8f;color:white;padding:8px 14px;border-radius:4px;font-size:11px;font-weight:700;">NB21 Synthesis</div>
    <div style="background:#1a7a6e;color:white;padding:8px 14px;border-radius:4px;font-size:12px;font-weight:700;">NB22 Adoption</div>
</div>
</div>

## Prerequisites

| Notebook | Topic | Why it matters here |
|----------|-------|---------------------|
| NB08 | EDD, MNND, DBSCAN | Utility metrics used throughout |
| NB17 | Adversarial experiments | Attack success rates cited in Part 1 |
| NB20 | Baseline comparison | Seven mechanisms and their EDD / attack results |
| NB21 | Research synthesis | Limitations and open questions extended here |

**Estimated reading time:** 30–40 minutes. No computation cells — all results are
drawn from NB17 and NB20.

---
## 22.1  The Better-Than-Nothing Baseline

The seven mechanisms in NB20 are all compared against each other, but none is
compared against the simplest alternative of all: doing nothing. Before asking
which mechanism is *best*, it is worth establishing what any mechanism achieves
relative to no protection.

### No-protection baseline

An adversary who holds both the original dataset and a set of unprotected coordinates
achieves 100 % success on the nearest-record spatial attack by definition — every
display coordinate *is* the original coordinate, so no inference is required.

From NB17 and NB20, even the weakest perturbation mechanism (uniform jitter ±62.5 m)
reduces this to approximately 40–87 % depending on geographic density. Donut
geomasking (50–125 m band), the most widely deployed approach in public-health
mapping practice (Hampton et al. 2010), reduces it to roughly 0.8 % on the dense
Soho dataset (NB20 Figure 20c).

| Protection level | Spatial attack success | Compound attack success | EDD |
|-----------------|----------------------|------------------------|-----|
| **None** | ~100 % | ~100 % | 0 m |
| **Donut geomasking** (50–125 m) | ~0.8 % | ~0 % | ~87 m |
| **Uniform jitter** (±62.5 m) | ~5.5 % | ~12 % | ~48 m |
| **Full pipeline** (global PRP) | ~0.2 % | ~0 % | ~8,976 km |

*All figures from NB20 on the 489-individual Soho cholera dataset.*

**The better-than-nothing argument is unambiguous for spatial re-identification:**
any of the seven evaluated mechanisms reduces spatial attack success from ~100 % to
well under 10 % on this dataset. The question is not whether to protect — it is which
protection is realistic to deploy.

### Why "realistic to deploy" matters

The geoprivacy literature has identified a persistent gap between the sophistication
of proposed mechanisms and actual practitioner adoption. Kounadi and Resch (2018)
found that over two decades of research on spatial re-identification risk, efforts to
instil sensitivity to the problem in researchers have been "relatively unsuccessful"
— practitioners ignore or are unaware of the risk when publishing point data.

Li (2025) identifies three interacting barriers: methodological complexity (mechanisms
are too technically demanding to implement without GIS programming expertise), cultural
and regulatory context (data protection norms vary across jurisdictions), and public
attitudes (privacy paradox — users report concern but do not act on it).

The implication for mechanism design: **a mechanism that is 5 % less effective but
10× simpler to deploy may provide more aggregate privacy protection across a
practitioner community than the theoretically optimal mechanism that few deploy.**

This is the adoption-weighted argument for simplicity — not a rejection of
cryptographic rigour, but a recognition that the metric that matters for population
privacy is *expected protection across all deployments*, not *maximum protection in
the best-case deployment*.

---
## 22.2  Complexity as a Third Axis

NB20 compares mechanisms on privacy and utility. This section adds a third axis:
**implementation complexity** — the effort required for a practitioner to move from
"I know this mechanism exists" to "I have deployed it on my dataset."

Complexity is multi-dimensional. The table below scores each of the seven NB20
mechanisms on four sub-dimensions using a three-level ordinal scale
(Low / Medium / High).

### Complexity rating table

| Mechanism | Implementation effort | Key management | Infrastructure dependency | Auditability | Overall |
|-----------|----------------------|----------------|--------------------------|--------------|---------|
| **Donut geomasking** | Low — one function, no external dependencies | None | None (runs locally) | High — displacement is inspectable | **Low** |
| **H3 hex-grid binning** | Low — `h3-py` library, one function | None | None (runs locally) | High — cell membership is inspectable | **Low** |
| **Uniform jitter** | Low — single random draw per record | None | None | High — displacement is inspectable | **Low** |
| **Gaussian perturbation** | Low — `np.random.normal`, one line | None | None | High | **Low** |
| **Planar Laplace** (geo-indistinguishability) | Medium — polar coordinate sampling, parameter selection requires understanding of epsilon | None | None | Medium — formal epsilon guarantee but epsilon choice is non-obvious | **Medium** |
| **Spatial cloaking** (k-NN centroid) | Medium — requires k-NN index, choice of k | None | Shared dataset required to compute centroids | Medium — k is inspectable but centroid computation is implicit | **Medium** |
| **Full PRP+AEAD+jitter pipeline** | High — Feistel PRP, ChaCha20-Poly1305 AEAD, HKDF-style key derivation, three subkeys | High — master key storage, rotation, access tier separation, incident response | Medium — key store or HSM required | Medium — tamper detection is strong but key provenance requires audit trail | **High** |

**Complexity sub-dimensions defined:**

- *Implementation effort*: lines of code, external dependencies, and expertise required
  to implement correctly (not just run an example).
- *Key management*: storage, rotation, access control, and incident response for
  cryptographic keys.
- *Infrastructure dependency*: whether deployment requires shared infrastructure
  (key stores, HSMs, shared databases) beyond a single researcher's workstation.
- *Auditability*: how easily a third party can verify that the protection was applied
  correctly and that the original data cannot be recovered from the protected output.

**Sources:** Complexity ratings are the authors' judgements informed by implementation
experience across NB00–NB21; methodological complexity as a barrier is documented in
Li (2025) and Kounadi and Resch (2018). The tooling gap for low-complexity mechanisms
is the direct motivation for tools such as MapSafe (Sharma et al. 2025).

---
## 22.3  The Three-Axis View

With privacy, utility, and complexity scored, it is possible to characterise each
mechanism's position in the three-axis space and articulate when each is the
right choice.

### Mechanism positions and deployment contexts

| Mechanism | Privacy | Utility | Complexity | Best deployment context |
|-----------|---------|---------|------------|------------------------|
| Donut geomasking | Medium (~0.8 % spatial attack) | Medium (~87 m EDD) | **Low** | Default choice for any public-health map publication; widely supported by existing tools (MapSafe, donutGeomask); appropriate when no key infrastructure exists |
| H3 hex-grid binning | Medium (aggregate-level) | Low–Medium (100–200 m EDD; AUC-L artefacts) | **Low** | Appropriate when aggregate spatial patterns are sufficient and individual precision is not needed; widely used in surveillance dashboards |
| Uniform jitter | Medium (~5.5 % spatial attack) | High (~48 m EDD) | **Low** | Appropriate as a fast research baseline; should not be the final protection layer for stigmatised populations |
| Gaussian perturbation | Medium (~6 % spatial attack) | High (~52 m EDD) | **Low** | No formal privacy guarantee (unlike Laplace); use only when Laplace implementation is not feasible |
| Planar Laplace | Medium (~6 % spatial attack) | High (~60 m EDD) | **Medium** | Preferred over Gaussian when a formal epsilon-geo-indistinguishability guarantee is required; appropriate for regulatory compliance contexts |
| Spatial cloaking | Medium–High (AUC-L artefacts; spatial attack ~2 %) | Low (100–200 m EDD; cluster collapse) | **Medium** | Appropriate when k-anonymity is a stated requirement; not recommended where spatial cluster analysis is needed downstream |
| Full pipeline | **High** (~0.2 % spatial attack; 0 % compound) | Low for display (~8,976 km EDD); lossless for decode | **High** | Appropriate when authorised decode access to exact coordinates is required alongside a publicly facing display tier; requires key infrastructure and organisational key custody |

### The adoption-weighted conclusion

The full pipeline occupies a unique position: it is the only mechanism that combines
lossless reversibility with near-zero display-tier re-identification. But it is also
the most complex to deploy correctly, and complexity is itself a privacy risk — a
mechanism misconfigured or not deployed is worse than a simpler mechanism deployed
universally.

The practical hierarchy for a public-health organisation without existing
cryptographic infrastructure:

1. **Start with donut geomasking or H3 binning** (Low complexity, meaningful protection,
   widely supported by existing tooling). This is better than nothing by a large margin.
2. **Add planar Laplace** if a formal privacy guarantee is needed for regulatory or
   publication requirements (Medium complexity, formal epsilon guarantee).
3. **Deploy the full pipeline** only when (a) authorised decode access to exact
   coordinates is an operational requirement, and (b) key management infrastructure
   can be sustained organisationally.

This hierarchy is consistent with the "Privacy by Design" framing of Kounadi and Resch
(2018): privacy protection should be proportionate to re-identification risk, data
sensitivity, and operational capacity — not maximised in the abstract.

---
## 22.4  Open Questions for the Field

The five questions from NB21.7 are reproduced and extended here with one additional
question motivated by the complexity analysis above.

**1. What is the acceptable re-identification risk threshold for public-health maps?**

The answer varies by health condition (cholera vs. overdose vs. respiratory), by
community context (rural vs. dense urban; see Old Naledi in NB16 vs. Philadelphia
in NB15), and by intended audience (clinicians, public, law enforcement). The
NB17 results show that even the dense Soho dataset has non-trivial k = 1 fractions
at fine QI granularity. No technical system can determine the acceptable threshold —
this is a policy decision that should involve affected communities.

**2. Can access-pattern privacy be added without sacrificing cross-session linkability?**

Per-session re-keying eliminates the tile-frequency side channel but also eliminates
the ability to link records across sessions — a common analytical requirement in
longitudinal surveillance. This is a fundamental tension, not an engineering oversight.
Oblivious RAM and private information retrieval are theoretical mitigations but are
not yet practical at public-health dataset scales.

**3. How does the pipeline perform against an adversary with auxiliary geographic data?**

NB17–18 model adversaries with access to the encrypted record database. A real-world
adversary may also hold commercial data (mobile phone pings, social media check-ins,
voter rolls) that increases re-identification risk beyond the formal model. The
compound attack in NB17 uses only the published QIs; auxiliary data would widen the
available quasi-identifier space.

**4. Is the privacy-utility frontier from NB20 stable across geographic contexts?**

All NB20 evaluation used the dense urban Soho cholera dataset. The Old Naledi TB
scenario (NB16–17) showed that settlement density strongly moderates jitter-only
spatial attack success (34.8 % vs. Philadelphia's 86.6 %). A systematic evaluation
across geographic contexts — rural, peri-urban, sparse — would be required to
generalise the NB20 findings.

**5. What epsilon value is appropriate for the display tier?**

Replacing bounded uniform jitter with planar Laplace provides a formal
epsilon-geo-indistinguishability guarantee. But selecting epsilon requires a decision
about acceptable probability ratios across locations, which is a substantive privacy
policy question. The npj Digital Medicine (2025) finding that strict DP (epsilon ≈ 1)
widens subgroup performance gaps in health AI suggests that epsilon calibration has
equity implications beyond the individual privacy-utility tradeoff.

**6. Does implementation complexity systematically predict non-adoption, and if so,
what is the threshold?** (New question from NB22 §22.2)

Li (2025) documents that high-complexity mechanisms are under-adopted, but the
relationship between specific complexity dimensions (key management, expertise,
infrastructure) and adoption rates has not been empirically studied across a
practitioner population. Knowing which sub-dimension is the binding constraint would
allow targeted tooling investment — for example, if key management is the binding
constraint, a managed key service would be a higher-return investment than simplifying
the cryptographic implementation.

---
## 22.5  Future Directions

| Direction | Description | Connection to this repo |
|-----------|-------------|------------------------|
| **Empirical adoption study** | Survey or observational study of actual geoprivacy mechanism choices by public-health practitioners; identify which complexity sub-dimensions are binding constraints | Motivated by NB22 §22.2 complexity rubric |
| **Differential privacy hybrids (NB23 planned)** | Combine planar Laplace (NB19) with PRP+AEAD pipeline for a mechanism with formal DP guarantee and reversibility | Addresses access-pattern leakage if noise is applied per-query |
| **Managed key service** | Reduce the key management complexity barrier by wrapping master key operations in a cloud KMS API; would move full pipeline complexity from High to Medium | Would allow NB22 Table §22.2 complexity rating to be revised |
| **Federated geospatial analytics (NB24 planned)** | Multi-party pipeline where no single party holds all keys; addresses the shared-infrastructure dependency in spatial cloaking | Requires multi-party key derivation and PRP composability |
| **Tooling ecosystem** | Extend the repo's `map_encryption` package with a one-function API mirroring donut geomasking's simplicity; lower the implementation effort sub-dimension | MapSafe (Sharma et al. 2025) demonstrates the model for desktop GIS integration |

---
## Key Takeaways

| Concept | What to remember |
|---------|-----------------|
| Better than nothing | Any evaluated mechanism reduces spatial re-identification from ~100 % (no protection) to under 10 % on the Soho dataset; the question is which is realistic to deploy |
| Complexity is a privacy risk | A mechanism not deployed provides no protection; adoption-weighted expected protection may favour simpler mechanisms even if their per-deployment performance is lower |
| Low-complexity tier | Donut geomasking, H3 hex-grid, uniform jitter: no key management, locally executable, inspectable — appropriate default for most public-health map publication |
| Medium-complexity tier | Planar Laplace, spatial cloaking: some implementation expertise required; Laplace adds formal epsilon guarantee; appropriate when regulatory compliance requires a formal privacy statement |
| High-complexity tier | Full PRP+AEAD+jitter pipeline: requires key management infrastructure and organisational key custody; appropriate only when reversibility and near-zero display-tier re-identification are simultaneously required |
| Equity implication | Strict DP (epsilon ≈ 1) may widen subgroup performance gaps (npj Digital Medicine 2025); epsilon calibration is a policy decision with equity implications |
| Open question | Whether complexity sub-dimensions predict non-adoption, and which sub-dimension is binding, has not been empirically studied |

## References

- **Hampton, K. H., Fitch, M. K., Allshouse, W. B., Doherty, I. A., Gesink, D. C., Leone, P. A., Serre, M. L., & Miller, W. C.** (2010). Mapping health data: improved privacy protection with donut method geomasking. *American Journal of Epidemiology, 172*(9), 1062–1069. https://doi.org/10.1093/aje/kwq248 — Canonical "simple method, widely adopted" benchmark; demonstrates that low-complexity geomasking substantially reduces re-identification risk relative to no protection.

- **Keßler, C., & McKenzie, G.** (2018). A geoprivacy manifesto. *Transactions in GIS, 22*(1), 3–19. https://doi.org/10.1111/tgis.12305 — 21 theses on why geoprivacy is a socio-technical challenge; most people neither understand nor control the underlying technologies; implicit adoption-barrier framing.

- **Kounadi, O., & Resch, B.** (2018). A geoprivacy by design guideline for research campaigns that use participatory sensing data. *Journal of Empirical Research on Human Research Ethics, 13*(3), 203–222. https://doi.org/10.1177/1556264618759877 — Practitioner-facing guidelines; documents that two decades of research on spatial re-identification risk have been "relatively unsuccessful" at changing researcher practice; frames complexity as an education and tooling problem.

- **Li, Z.** (2025). Challenges in geoprivacy protection: methodological issues, cultural and regulatory contexts, and public attitudes. *Transactions in GIS.* https://doi.org/10.1111/tgis.70075 — Identifies three interacting adoption barriers: methodological complexity, cultural/regulatory context, and privacy paradox; notes some near-optimal mechanisms take 1–2 hours to compute per user profile.

- **Riahi, H. K., et al.** (2025). Differential privacy for medical deep learning: methods, tradeoffs, and deployment implications. *npj Digital Medicine.* https://doi.org/10.1038/s41746-025-02280-z — Documents that strict DP (epsilon ≈ 1) causes substantial accuracy loss and widens subgroup performance gaps; practical argument that epsilon calibration is a deployment barrier with equity implications.

- **Sharma, P. N., et al.** (2025). MapSafe QGIS plugin: a complete open-source geoprivacy tool for desktop GIS applications. *Journal of Location Based Services.* https://doi.org/10.1080/17489725.2025.2497752 — Directly motivated by the tooling gap reducing practitioner adoption; integrates donut masking and hex binning into QGIS to lower the implementation barrier.

- **Andrés, M. E., Bordenabe, N. E., Chatzikokolakis, K., & Palamidessi, C.** (2013). Geo-indistinguishability: differential privacy for location-based systems. *Proceedings of ACM CCS 2013*, 901–914. https://doi.org/10.1145/2508859.2516735 — Planar Laplace mechanism; formal epsilon-geo-indistinguishability guarantee; medium-complexity tier in NB22 §22.2.

- **Lin, Y.** (2023). Geo-indistinguishable masking: enhancing privacy protection in spatial point mapping. *Cartography and Geographic Information Science.* https://doi.org/10.1080/15230406.2023.2267967 — Source of the EDD and AUC-L evaluation metrics used in NB08, NB12, NB13, NB20, and cited results in this notebook.